## Converter GT para PNG

Seleciona uma mascara aleatoria em data/Ground-of-truth, salva em PNG em generated/Ground-of-truth, binariza, exibe lado a lado e aplica as mascaras na foto original.

### Parte 1 - Selecionar e converter para PNG

In [ ]:
import os
import random

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

BASE_DIR = os.getcwd()
GROUND_TRUTH_DIR = os.path.join(BASE_DIR, 'data', 'Ground-of-truth')
ORIGINAL_DIR = os.path.join(BASE_DIR, 'data', 'Original')
OUTPUT_DIR = os.path.join(BASE_DIR, 'generated', 'Ground-of-truth-referencia')

LIMIAR_BINARIZACAO = 127

if not os.path.isdir(GROUND_TRUTH_DIR):
    raise FileNotFoundError(f'Diretorio nao encontrado: {GROUND_TRUTH_DIR}')

if not os.path.isdir(ORIGINAL_DIR):
    raise FileNotFoundError(f'Diretorio nao encontrado: {ORIGINAL_DIR}')

os.makedirs(OUTPUT_DIR, exist_ok=True)

arquivos = [
    nome for nome in os.listdir(GROUND_TRUTH_DIR)
    if nome.lower().endswith('.jpg')
]

if not arquivos:
    raise FileNotFoundError(f'Nenhum .jpg encontrado em {GROUND_TRUTH_DIR}')

nome_arquivo = random.choice(arquivos)
caminho_original = os.path.join(GROUND_TRUTH_DIR, nome_arquivo)
nome_base, _ = os.path.splitext(nome_arquivo)
caminho_saida = os.path.join(OUTPUT_DIR, f'{nome_base}.png')

with Image.open(caminho_original) as img:
    img.save(caminho_saida, format='PNG')

print(f'Original: {caminho_original}')
print(f'PNG salvo: {caminho_saida}')


### Parte 2 - Binarizar o PNG gerado

In [ ]:
with Image.open(caminho_saida) as img_png:
    img_gray = img_png.convert('L')
    matriz = np.array(img_gray)
    matriz_binarizada = np.where(matriz > LIMIAR_BINARIZACAO, 255, 0).astype(np.uint8)
    img_binarizada = Image.fromarray(matriz_binarizada, mode='L')
    img_binarizada.save(caminho_saida, format='PNG')

print(f'PNG binarizado salvo: {caminho_saida}')


### Parte 3 - Exibir JPG original e PNG binarizado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

with Image.open(caminho_original) as img_jpg:
    axes[0].imshow(img_jpg, cmap='gray')
    axes[0].set_title('JPG Original')
    axes[0].axis('off')

with Image.open(caminho_saida) as img_png:
    axes[1].imshow(img_png, cmap='gray')
    axes[1].set_title('PNG Binarizado')
    axes[1].axis('off')

plt.tight_layout()
plt.show()


### Parte 4 - Aplicar mascaras na foto original

In [ ]:
caminho_foto_original = os.path.join(ORIGINAL_DIR, f'{nome_base}.jpg')
if not os.path.isfile(caminho_foto_original):
    raise FileNotFoundError(f'Foto original nao encontrada: {caminho_foto_original}')

with Image.open(caminho_foto_original) as foto_original:
    foto_rgb = foto_original.convert('RGB')
    fundo_preto = Image.new('RGB', foto_rgb.size, (0, 0, 0))

with Image.open(caminho_original) as mascara_jpg:
    mascara_jpg_gray = mascara_jpg.convert('L')
    matriz_jpg = np.array(mascara_jpg_gray)
    mask_jpg = Image.fromarray(np.where(matriz_jpg > LIMIAR_BINARIZACAO, 255, 0).astype(np.uint8), mode='L')

with Image.open(caminho_saida) as mascara_png:
    mascara_png_gray = mascara_png.convert('L')
    matriz_png = np.array(mascara_png_gray)
    mask_png = Image.fromarray(np.where(matriz_png > LIMIAR_BINARIZACAO, 255, 0).astype(np.uint8), mode='L')

resultado_jpg = Image.composite(foto_rgb, fundo_preto, mask_jpg)
resultado_png = Image.composite(foto_rgb, fundo_preto, mask_png)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(resultado_jpg)
axes[0].set_title('Mascara JPG aplicada')
axes[0].axis('off')

axes[1].imshow(resultado_png)
axes[1].set_title('Mascara PNG binarizada aplicada')
axes[1].axis('off')

plt.tight_layout()
plt.show()
